# CP3 · Notebook 05 — Consistencia

¿El modelo es **determinista**? Con `do_sample=False` (greedy) **debería** dar la misma respuesta cada vez. Con sampling (`temperature>0`) **varía**.

Vamos a verificarlo con 3 imágenes × 6 llamadas. ~4 min.

In [ ]:
import time, json
from pathlib import Path
import torch
from PIL import Image
from transformers import AutoModelForCausalLM, AutoTokenizer

DATA = Path('../datasets/dashcam-curated')
OUT = Path('../outputs')

MODEL_ID = 'vikhyatk/moondream2'
REVISION = '2024-08-26'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=REVISION, trust_remote_code=True,
                                               torch_dtype=torch.float32).eval()
print('modelo cargado')

## 1. Seleccionar 3 imágenes para el test

Una `trivial`, una `urbano_standard` y una `edge_semantic`.

In [ ]:
test_imgs = []
for cat in ['trivial', 'urbano_standard', 'edge_semantic']:
    cat_imgs = sorted((DATA / cat).glob('*.jpg')) + sorted((DATA / cat).glob('*.png'))
    if cat_imgs:
        test_imgs.append((cat, cat_imgs[0]))

for cat, p in test_imgs:
    print(f'  {cat:20s} {p.name}')

## 2. Test greedy (3 imágenes × 3 calls = 9 inferencias)

Moondream `answer_question` por defecto usa greedy decoding. Las 3 respuestas DEBERÍAN ser idénticas.

In [ ]:
PROMPT = 'Describe this driving scene in 2 sentences.'

greedy_results = []
for cat, img_path in test_imgs:
    img = Image.open(img_path).convert('RGB')
    enc = model.encode_image(img)
    runs = []
    for i in range(3):
        ans = model.answer_question(enc, PROMPT, tokenizer)
        runs.append(ans.strip())
    n_distinct = len(set(runs))
    greedy_results.append({'category': cat, 'name': img_path.name, 'runs': runs, 'distinct': n_distinct})
    print(f'\n=== {cat} / {img_path.name} ===')
    print(f'Respuestas distintas: {n_distinct}/3  ({"determinista" if n_distinct==1 else "VARIABLE — inesperado"})')
    for i, r in enumerate(runs):
        print(f'  run {i+1}: {r[:200]}...' if len(r) > 200 else f'  run {i+1}: {r}')

## 3. Test sampling (`do_sample=True`, temperature=0.7)

Nota: `model.answer_question` no expone `temperature` directamente. Usamos `generate` con kwargs.

**Esperamos**: respuestas variables — diferentes a cada call.

In [ ]:
# answer_question internamente usa model.generate. Aproximamos sampling con generate directamente.
# Si tu versión de Moondream tiene API distinta, ajustar.

def sample_answer(img_pil, prompt, temperature=0.7, max_new_tokens=120):
    enc = model.encode_image(img_pil)
    # Truco: encode_image + answer_question internamente generan con greedy.
    # Para sampling tenemos que generar manualmente. La estructura interna depende de la versión.
    # Ver moondream2 README: el método 'answer_question' admite **generate_kwargs en algunas versiones.
    try:
        ans = model.answer_question(
            enc, prompt, tokenizer,
            do_sample=True, temperature=temperature, top_p=0.9,
            max_new_tokens=max_new_tokens,
        )
        return ans.strip()
    except TypeError:
        # versión anterior — no admite kwargs en answer_question
        print('  ⚠️  esta versión de Moondream no expone sampling kwargs en answer_question.')
        print('     Reporta esto en la entrega como una limitación práctica observada.')
        return model.answer_question(enc, prompt, tokenizer).strip()

sampling_results = []
for cat, img_path in test_imgs:
    img = Image.open(img_path).convert('RGB')
    runs = []
    for i in range(3):
        ans = sample_answer(img, PROMPT, temperature=0.7)
        runs.append(ans)
    sampling_results.append({'category': cat, 'name': img_path.name, 'runs': runs, 'distinct': len(set(runs))})
    print(f'\n=== {cat} / {img_path.name} === (sampling t=0.7)')
    print(f'Respuestas distintas: {len(set(runs))}/3')
    for i, r in enumerate(runs):
        print(f'  run {i+1}: {r[:200]}...' if len(r) > 200 else f'  run {i+1}: {r}')

## 4. Implicaciones para auto-labeling

Si auto-etiquetas con un VLM (P5):

- **Greedy es reproducible** → seguro para baseline.
- **Sampling introduce variabilidad** → útil para **ensemble** o **explorar alternativas**, pero **peligroso si no controlas la varianza**.
- **Truco práctico**: pedir 3 respuestas con sampling, quedarse con la **mayoría** (voting). Reduce varianza, mejor calidad.

Apunta tu observación para `07_local_vs_frontier.md`.

In [ ]:
with open(OUT / '05_consistency.json', 'w') as f:
    json.dump({'greedy': greedy_results, 'sampling': sampling_results}, f, indent=2, ensure_ascii=False)
print(f'✅ Guardado en {OUT / "05_consistency.json"}')
print('Ve a 06_failure_analysis.ipynb')